# Módulo A.1 — Dataset geoespacial

Se arma la tabla maestra con datos reales:

- **Presencias de té**: registros de *Camellia sinensis* obtenidos desde GBIF.
- **Variables ambientales**: datos bioclimáticos y de elevación provenientes de WorldClim 2.1.
- **Suelo**: el pH del suelo se ha obtenido de SoilGrids.


NOTA sobre el suelo: actualmente, la API REST de SoilGrids está fuera de servicio temporalmente, así que he tenido que obtener el pH directamente de sus archivos GeoTIFF públicos (files.isric.org). Es importante mencionar que SoilGrids utiliza la proyección Homolosine, por lo que las coordenadas se transforman antes de realizar el muestreo.

**SELECCIÓN DE VARIABLES**: se han elegido variables bioclimáticas que están alineadas con la literatura sobre la idoneidad del té (Bania et al., 2025; Hajiboland, 2017) y que están pensadas para el clima tropical isohídrico de Costa Rica. Se da prioridad a la isotermalidad y al rango diurno, ya que son más informativos en climas sin estaciones térmicas marcadas, así como a la precipitación del trimestre seco, que indica el estrés hídrico durante la estación seca.

## 1. Instalar librerías

In [1]:
!pip install rasterio -q   # rasterio sirve para leer y muestrear los mapas (rasters) de WorldClim

## 2. Montar Drive

In [2]:
import os                       # comprobar rutas y estado del Drive
from pathlib import Path        # manejar rutas de forma segura

# Montar el Drive solo si aún no lo está (evita montarlo dos veces)
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Carpeta del proyecto y subcarpeta para los mapas de WorldClim
CARPETA_BASE = Path('/content/drive/MyDrive/TFM_TeaSuitability')
CARPETA_BASE.mkdir(parents=True, exist_ok=True)   # crea la carpeta si no existe
CARPETA_CLIMA = CARPETA_BASE / 'worldclim'
CARPETA_CLIMA.mkdir(parents=True, exist_ok=True)  # crea la subcarpeta de clima
print('Carpeta del proyecto:', CARPETA_BASE)

Mounted at /content/drive
Carpeta del proyecto: /content/drive/MyDrive/TFM_TeaSuitability


## 3. Importaciones

In [3]:
import requests              # llamadas a la API de GBIF
import zipfile               # descomprimir los .zip de WorldClim
import urllib.request        # descargar archivos por URL
import numpy as np           # cálculo numérico
import pandas as pd          # tablas de datos
import rasterio              # leer y muestrear rasters
from rasterio.warp import transform as warp_transform  # transformar coordenadas

## 4. Funciones para las presencias de té (GBIF)

In [4]:
def obtener_taxon_key(nombre_especie):
    """
    Obtiene el identificador taxonómico (taxonKey) de una especie en GBIF.

    Parameters
    ----------
    nombre_especie : str
        Nombre científico de la especie (p. ej. 'Camellia sinensis').

    Returns
    -------
    int
        El taxonKey que GBIF asigna a la especie.

    Raises
    ------
    ValueError
        Si GBIF no devuelve una clave para el nombre indicado.

    Examples
    --------
    >>> clave = obtener_taxon_key('Camellia sinensis')   # doctest: +SKIP
    >>> isinstance(clave, int)                            # doctest: +SKIP
    True
    """
    url = 'https://api.gbif.org/v1/species/match'
    info = requests.get(url, params={'name': nombre_especie}).json()
    # se intenta con la clave principal y, si no, con la aceptada
    clave = info.get('usageKey') or info.get('acceptedUsageKey')
    if clave is None:
        raise ValueError(f'GBIF no devolvió taxonKey para: {nombre_especie}')
    return clave


def descargar_presencias_gbif(clave_taxon, objetivo=800, paso=300):
    """
    Descarga coordenadas de presencia de una especie desde GBIF.

    Recorre la API de ocurrencias paginando y se queda solo con los puntos
    que tienen coordenadas válidas, sin nulos ni duplicados.

    Parameters
    ----------
    clave_taxon : int
        Identificador taxonómico (taxonKey) de la especie en GBIF.
    objetivo : int, optional
        Número máximo de presencias a descargar. Por defecto 800.
    paso : int, optional
        Registros por página (GBIF permite hasta 300). Por defecto 300.

    Returns
    -------
    pandas.DataFrame
        Tabla con columnas 'lon' y 'lat' de las presencias.

    Examples
    --------
    >>> pres = descargar_presencias_gbif(2874863)   # doctest: +SKIP
    >>> {'lon', 'lat'}.issubset(pres.columns)        # doctest: +SKIP
    True
    """
    url = 'https://api.gbif.org/v1/occurrence/search'
    registros = []
    offset = 0
    while len(registros) < objetivo:
        respuesta = requests.get(url, params={
            'taxonKey': clave_taxon,
            'hasCoordinate': 'true',        # solo registros con lon/lat
            'hasGeospatialIssue': 'false',  # descarta coordenadas problemáticas
            'limit': paso, 'offset': offset,
        }).json()
        registros.extend(respuesta['results'])
        offset += paso
        if respuesta['endOfRecords']:       # no hay más páginas
            break
    # se extraen lon/lat, se quitan nulos y duplicados
    return pd.DataFrame([
        {'lon': r.get('decimalLongitude'), 'lat': r.get('decimalLatitude')}
        for r in registros
        if r.get('decimalLongitude') is not None
        and r.get('decimalLatitude') is not None
    ]).drop_duplicates().reset_index(drop=True)

## 5. Función para descargar WorldClim

In [5]:
def descargar_worldclim(carpeta, resolucion='5m'):
    """
    Descarga y descomprime las variables de WorldClim 2.1.

    Baja las 19 variables bioclimáticas y la elevación en la resolución
    indicada, y las extrae como archivos .tif en la carpeta dada.

    Parameters
    ----------
    carpeta : pathlib.Path
        Carpeta donde se guardarán y extraerán los archivos.
    resolucion : str, optional
        Resolución espacial: '10m', '5m', '2.5m' o '30s'. Por defecto '5m'.

    Returns
    -------
    None
        No devuelve nada; deja los .tif en la carpeta.
    """
    carpeta.mkdir(parents=True, exist_ok=True)   # asegura que la carpeta existe
    base = 'https://geodata.ucdavis.edu/climate/worldclim/2_1/base/'
    archivos = [f'wc2.1_{resolucion}_bio.zip', f'wc2.1_{resolucion}_elev.zip']
    for archivo in archivos:
        destino = carpeta / archivo
        if not destino.exists():                 # no re-descargar si ya está
            print('Descargando', archivo, '... (puede tardar)')
            urllib.request.urlretrieve(base + archivo, str(destino))
        with zipfile.ZipFile(destino) as z:      # extraer los .tif
            z.extractall(carpeta)
    print('WorldClim listo en:', carpeta)

## 6a. Funciones geoespaciales (pseudo-ausencias, muestreo y ensamblado)

In [6]:
def generar_pseudoausencias(positivos, n_negativos, limites,
                            distancia_minima=0.5, semilla=42):
    """
    Genera puntos negativos aleatorios lejos de las presencias reales.

    Parameters
    ----------
    positivos : pandas.DataFrame
        Tabla con columnas 'lon' y 'lat' de las presencias.
    n_negativos : int
        Número de pseudo-ausencias a generar.
    limites : tuple of float
        Rectángulo de muestreo (lon_min, lat_min, lon_max, lat_max).
    distancia_minima : float, optional
        Distancia mínima (grados) a cualquier presencia. Por defecto 0.5.
    semilla : int, optional
        Semilla aleatoria para reproducibilidad. Por defecto 42.

    Returns
    -------
    pandas.DataFrame
        Tabla con columnas 'lon' y 'lat' de los negativos.

    Raises
    ------
    ValueError
        Si n_negativos es negativo.

    Examples
    --------
    >>> pos = pd.DataFrame({'lon': [-83.6], 'lat': [9.9]})
    >>> neg = generar_pseudoausencias(pos, 3, (-86, 8, -82, 11))
    >>> len(neg) <= 3
    True
    """
    if n_negativos < 0:                          # validación de entrada
        raise ValueError(f'n_negativos debe ser >= 0, recibido: {n_negativos}')
    generador = np.random.default_rng(semilla)   # azar reproducible
    lon_min, lat_min, lon_max, lat_max = limites
    negativos = []
    lon_pos = positivos['lon'].to_numpy()
    lat_pos = positivos['lat'].to_numpy()
    intentos = 0
    while len(negativos) < n_negativos and intentos < n_negativos * 200:
        intentos += 1
        lon = generador.uniform(lon_min, lon_max)
        lat = generador.uniform(lat_min, lat_max)
        d = np.sqrt((lon_pos - lon)**2 + (lat_pos - lat)**2)  # distancia a positivos
        if d.min() >= distancia_minima:          # solo si está lejos de todos
            negativos.append({'lon': lon, 'lat': lat})
    return pd.DataFrame(negativos)


def muestrear(ruta, coordenadas):
    """
    Extrae el valor de un raster en cada coordenada (lon, lat).

    Parameters
    ----------
    ruta : str or pathlib.Path
        Ruta al archivo raster (.tif).
    coordenadas : list of tuple of float
        Lista de pares (lon, lat). Primero longitud, luego latitud.

    Returns
    -------
    list of float
        Un valor por coordenada; NaN si el punto cae fuera de cobertura.
    """
    valores = []
    with rasterio.open(ruta) as raster:          # abre y cierra el raster solo
        for celda in raster.sample(coordenadas):  # OJO: espera (lon, lat)
            dato = float(celda[0])
            if raster.nodata is not None and dato == raster.nodata:
                dato = np.nan                      # sin dato -> faltante
            valores.append(dato)
    return valores


def construir_dataset(presencias, negativos, capas):
    """
    Une presencias y negativos y les añade las variables ambientales.

    Muestrea cada capa raster en todos los puntos, elimina los que caen
    fuera de cobertura (mar) y reequilibra las clases a ~1:1.

    Parameters
    ----------
    presencias : pandas.DataFrame
        Presencias reales (columnas 'lon', 'lat').
    negativos : pandas.DataFrame
        Pseudo-ausencias (columnas 'lon', 'lat').
    capas : dict of {str: pathlib.Path}
        Diccionario {nombre_variable: ruta_del_raster}.

    Returns
    -------
    pandas.DataFrame
        Dataset con lon, lat, clase y una columna por variable.
    """
    pres = presencias.copy(); pres['clase'] = 1   # té = 1
    neg = negativos.copy();  neg['clase'] = 0     # no té = 0
    dataset = pd.concat([pres, neg], ignore_index=True)

    coords = list(zip(dataset['lon'], dataset['lat']))
    for nombre, ruta in capas.items():            # una columna por capa
        dataset[nombre] = muestrear(ruta, coords)

    dataset = dataset.dropna().reset_index(drop=True)  # quita puntos sin datos

    # reequilibrar a ~1:1 (se generaron negativos de más)
    n_pos = int((dataset['clase'] == 1).sum())
    pos_df = dataset[dataset['clase'] == 1]
    neg_df = dataset[dataset['clase'] == 0].sample(
        n=min(n_pos, (dataset['clase'] == 0).sum()), random_state=42)
    return pd.concat([pos_df, neg_df], ignore_index=True)

## 6b. Función para el pH del suelo (SoilGrids)

In [7]:
import os

# Configuración de red para lecturas remotas más estables (timeouts y reintentos)
os.environ['GDAL_HTTP_TIMEOUT'] = '30'          # segundos máximo por petición
os.environ['GDAL_HTTP_MAX_RETRY'] = '3'         # reintentos si falla
os.environ['GDAL_HTTP_RETRY_DELAY'] = '2'       # espera entre reintentos
os.environ['CPL_VSIL_CURL_USE_HEAD'] = 'NO'     # evita peticiones extra

# URL del mapa de pH de SoilGrids (media, capa superficial 0-5 cm).
URL_PH = ('/vsicurl/https://files.isric.org/soilgrids/latest/data/'
          'phh2o/phh2o_0-5cm_mean.vrt')


def muestrear_ph_soilgrids(lons, lats, url=URL_PH, tam_lote=200):
    """
    Obtiene el pH del suelo (SoilGrids) en cada coordenada (lon, lat).

    Lee de forma remota, POR LOTES y con reintentos, para ser estable con el
    servidor de SoilGrids. Transforma las coordenadas a la proyección
    Homolosine de SoilGrids antes de muestrear. Los valores vienen x10.

    Parameters
    ----------
    lons, lats : array-like of float
        Longitudes y latitudes de los puntos (EPSG:4326).
    url : str, optional
        Ruta al GeoTIFF/VRT de pH de SoilGrids.
    tam_lote : int, optional
        Cuántos puntos leer por lote (para mostrar progreso). Por defecto 200.

    Returns
    -------
    list of float
        pH del suelo en cada punto (NaN si no hay dato).
    """
    valores = []
    with rasterio.open(url) as src:
        # transformar lon/lat -> proyección de SoilGrids (Homolosine)
        xs, ys = warp_transform('EPSG:4326', src.crs, list(lons), list(lats))
        pares = list(zip(xs, ys))
        total = len(pares)
        # procesar por lotes para ver el progreso y no colgarse en silencio
        for inicio in range(0, total, tam_lote):
            lote = pares[inicio:inicio + tam_lote]
            for celda in src.sample(lote):
                v = float(celda[0])
                if src.nodata is not None and v == src.nodata:
                    valores.append(np.nan)     # sin dato -> faltante
                else:
                    valores.append(v / 10.0)   # SoilGrids da el pH x10
            print(f'  pH muestreado: {min(inicio + tam_lote, total)}/{total}')
    return valores

## 7. Ejecución: presencias de té

In [8]:
clave = obtener_taxon_key('Camellia sinensis')
print('taxonKey:', clave)
presencias = descargar_presencias_gbif(clave, objetivo=800)
print('Presencias de té obtenidas:', len(presencias))
presencias.head()

taxonKey: 3189635
Presencias de té obtenidas: 782


,lon,lat
0,118.013054,29.941719
1,47.241323,-20.521527
2,121.503180,25.163700
3,121.741242,24.805763
4,120.603078,23.564246


## 8. Ejecución: descargar WorldClim

In [9]:
descargar_worldclim(CARPETA_CLIMA, resolucion='5m')

Descargando wc2.1_5m_bio.zip ... (puede tardar)
Descargando wc2.1_5m_elev.zip ... (puede tardar)
WorldClim listo en: /content/drive/MyDrive/TFM_TeaSuitability/worldclim


## 9s. Ejecución: pseudo-ausencias y construcción del dataset

In [10]:
# rectángulo = extensión geográfica de las presencias
limites = (presencias['lon'].min(), presencias['lat'].min(),
           presencias['lon'].max(), presencias['lat'].max())
# el doble de negativos (algunos caerán en el mar y se descartarán)
negativos = generar_pseudoausencias(presencias, len(presencias) * 2, limites)

CAPAS = {
    # variable descriptiva -> archivo WorldClim (nº de la variable bioclimática)
    'temperatura_media':      CARPETA_CLIMA / 'wc2.1_5m_bio_1.tif',   # bio1: temperatura media anual
    'rango_diurno':           CARPETA_CLIMA / 'wc2.1_5m_bio_2.tif',   # bio2: oscilación media día-noche
    'precipitacion_anual':    CARPETA_CLIMA / 'wc2.1_5m_bio_12.tif',  # bio12: lluvia total anual
    'estacionalidad_precip':  CARPETA_CLIMA / 'wc2.1_5m_bio_15.tif',  # bio15: variación de lluvia entre estaciones
    'precip_trimestre_seco':  CARPETA_CLIMA / 'wc2.1_5m_bio_17.tif',  # bio17: lluvia en la estación seca
    'elevacion':              CARPETA_CLIMA / 'wc2.1_5m_elev.tif',    # altitud del terreno
}

dataset = construir_dataset(presencias, negativos, CAPAS)
print('Dataset real construido:', dataset.shape)
print('Positivos:', int((dataset['clase']==1).sum()),
      '| Negativos:', int((dataset['clase']==0).sum()))
dataset.head()

Dataset real construido: (1333, 9)
Positivos: 780 | Negativos: 553


,lon,lat,clase,temperatura_media,rango_diurno,precipitacion_anual,estacionalidad_precip,precip_trimestre_seco,elevacion
0,118.013054,29.941719,1,14.924833,7.618333,1670.0,54.601158,158.0,429.0
1,47.241323,-20.521527,1,16.620459,11.176250,1437.0,87.080925,72.0,1472.0
2,121.503180,25.163700,1,20.207001,5.704167,3005.0,26.002686,554.0,256.0
3,121.741242,24.805763,1,20.652042,5.553583,3160.0,35.268108,509.0,297.0
4,120.603078,23.564246,1,19.789417,5.974333,2526.0,91.668259,76.0,748.0


## 9b. Añadir el pH del suelo (SoilGrids)

Se mide el pH en cada uno de los puntos del dataset que ya hemos creado. Dado que la lectura es remota, puede que tome un poco de tiempo. Para los pocos valores que falten, se sustituyen con la mediana, así no se pierde ninguna fila.

In [11]:
# muestrear el pH en cada punto (lon, lat) del dataset
ph = muestrear_ph_soilgrids(dataset['lon'].to_numpy(), dataset['lat'].to_numpy())
dataset['ph_suelo'] = ph

# rellenar los pocos faltantes con la mediana (imputación simple)
dataset['ph_suelo'] = dataset['ph_suelo'].fillna(dataset['ph_suelo'].median())

print('pH del suelo añadido. Estadísticos:')
print(dataset['ph_suelo'].describe().round(2))
print('Faltantes de pH tras imputar:', int(dataset['ph_suelo'].isna().sum()))

  pH muestreado: 200/1333
  pH muestreado: 400/1333
  pH muestreado: 600/1333
  pH muestreado: 800/1333
  pH muestreado: 1000/1333
  pH muestreado: 1200/1333
  pH muestreado: 1333/1333
pH del suelo añadido. Estadísticos:
count    1333.00
mean        5.86
std         1.01
min         3.80
25%         5.20
50%         5.50
75%         6.20
max         8.90
Name: ph_suelo, dtype: float64
Faltantes de pH tras imputar: 0


## 10. Guardar el dataset real

In [12]:
RUTA_CSV = CARPETA_BASE / 'dataset_master.csv'
dataset.to_csv(RUTA_CSV, index=False)
print('GUARDADO en:', RUTA_CSV)
print('Forma:', dataset.shape)
print('Variables:', [c for c in dataset.columns if c not in ('lon','lat','clase')])

# comprobación final: confirmar que el archivo existe en el Drive
print('¿Archivo creado?', RUTA_CSV.exists())

GUARDADO en: /content/drive/MyDrive/TFM_TeaSuitability/dataset_master.csv
Forma: (1333, 10)
Variables: ['temperatura_media', 'rango_diurno', 'precipitacion_anual', 'estacionalidad_precip', 'precip_trimestre_seco', 'elevacion', 'ph_suelo']
¿Archivo creado? True
